# 14 - Privacy Transformations

Este notebook testa transformações inspiradas na literatura sobre privacidade em ECG:
- normalization;
- noise injection;
- quantization;
- PCA.

A comparação usa sempre os mesmos segmentos e o mesmo split por paciente para utilidade e linkability.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import FINAL_SEGMENT_FEATURES_DIR, OUTPUTS_TABLES_DIR
from modeling import (
    split_segments_by_patient_stratified,
    evaluate_utility_model_on_split,
    run_linkability_baselines_on_split,
)
from privacy_transforms import fit_transform_feature_split

## Settings

In [2]:
MAX_CHUNKS = 20  # None for full dataset
RANDOM_STATE = 42
TEST_SIZE = 0.2

LINK_MAX_PAIRS = 2000
LINK_MIN_SEGMENT_GAP = 4
LINK_MAX_POSITIVE_PAIRS_PER_PATIENT = 3

# Focused comparison between PCA and random projection.
TRANSFORM_EXPERIMENTS = [
    {"name": "identity", "method": "identity", "params": {}},
    {"name": "pca_120", "method": "pca", "params": {"n_components": 120, "random_state": RANDOM_STATE}},
    {"name": "rp_60", "method": "random_projection", "params": {"n_components": 60, "random_state": RANDOM_STATE}},
    {"name": "rp_100", "method": "random_projection", "params": {"n_components": 100, "random_state": RANDOM_STATE}},
    {"name": "rp_120", "method": "random_projection", "params": {"n_components": 120, "random_state": RANDOM_STATE}},
    {"name": "rp_150", "method": "random_projection", "params": {"n_components": 150, "random_state": RANDOM_STATE}},
]


## Load Final Segment Dataset

In [3]:
manifest = json.loads((FINAL_SEGMENT_FEATURES_DIR / "manifest.json").read_text(encoding="utf-8"))
chunk_files = [FINAL_SEGMENT_FEATURES_DIR / item["chunk_file"] for item in manifest["chunks"]]
if MAX_CHUNKS is not None:
    chunk_files = chunk_files[:MAX_CHUNKS]

features_df = pd.concat(
    [pd.read_csv(chunk_file, compression="gzip", low_memory=False) for chunk_file in chunk_files],
    ignore_index=True,
)

print("Window (s):", manifest["window_sec"])
print("Step (s):", manifest["step_sec"])
print("Chunk files loaded:", len(chunk_files))
print("Features dataframe:", features_df.shape)

Window (s): 2.0
Step (s): 1.0
Chunk files loaded: 20
Features dataframe: (45000, 215)


## Fixed Split by Patient

O mesmo split por paciente é reutilizado para todas as transformações.

In [4]:
train_df, test_df = split_segments_by_patient_stratified(
    features_df,
    group_col="patient_id",
    label_col="utility_label",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print("Train segments:", train_df.shape)
print("Test segments:", test_df.shape)
print("Train patients:", train_df['patient_id'].nunique())
print("Test patients:", test_df['patient_id'].nunique())

Train segments: (36000, 215)
Test segments: (9000, 215)
Train patients: 4000
Test patients: 1000


## Run Transform Experiments

In [5]:
results_rows = []

for experiment in TRANSFORM_EXPERIMENTS:
    name = experiment["name"]
    method = experiment["method"]
    params = experiment["params"]

    train_transformed, test_transformed, fitted = fit_transform_feature_split(
        train_df=train_df,
        test_df=test_df,
        method=method,
        **params,
    )

    utility_out = evaluate_utility_model_on_split(
        train_df=train_transformed,
        test_df=test_transformed,
        model_name="LogisticRegression",
        target_col="utility_label",
    )
    utility_eval = utility_out["evaluation"]

    linkability_out = run_linkability_baselines_on_split(
        train_df=train_transformed,
        test_df=test_transformed,
        max_pairs=LINK_MAX_PAIRS,
        representation="absdiff",
        random_state=RANDOM_STATE,
        min_segment_gap=LINK_MIN_SEGMENT_GAP,
        max_positive_pairs_per_patient=LINK_MAX_POSITIVE_PAIRS_PER_PATIENT,
        include_distance_baselines=False,
    )
    linkability_summary = linkability_out["summary_df"]
    xgb_row = linkability_summary[linkability_summary["model"] == "XGBoost"].iloc[0]

    results_rows.append(
        {
            "transform_name": name,
            "method": method,
            "n_output_features": len(fitted.output_feature_names),
            "utility_f1": utility_eval["f1_score"],
            "utility_balanced_accuracy": utility_eval["balanced_accuracy"],
            "utility_roc_auc": utility_eval["roc_auc"],
            "utility_pr_auc": utility_eval["pr_auc"],
            "linkability_f1": xgb_row["f1_score"],
            "linkability_roc_auc": xgb_row["roc_auc"],
            "linkability_pr_auc": xgb_row["pr_auc"],
        }
    )

privacy_transform_results_df = pd.DataFrame(results_rows)
privacy_transform_results_df

,transform_name,method,n_output_features,utility_f1,utility_balanced_accuracy,utility_roc_auc,utility_pr_auc,linkability_f1,linkability_roc_auc,linkability_pr_auc
0,identity,identity,208,0.723567,0.854232,0.930469,0.794538,0.983976,0.998381,0.998482
1,winsor_01_99,winsorization,208,0.729752,0.857262,0.931998,0.798535,0.983237,0.998797,0.998850
2,winsor_05_95,winsorization,208,0.717855,0.848457,0.929945,0.798970,0.983483,0.998587,0.998635
3,winsor_10_90,winsorization,208,0.721771,0.851521,0.929105,0.793556,0.983229,0.998367,0.998413
4,pca_120,pca,120,0.714832,0.847961,0.927374,0.790344,0.929126,0.982065,0.983659


## Compare Against Identity

In [6]:
baseline_row = privacy_transform_results_df.loc[
    privacy_transform_results_df["transform_name"] == "identity"
].iloc[0]

comparison_df = privacy_transform_results_df.copy()
comparison_df["delta_utility_f1"] = comparison_df["utility_f1"] - baseline_row["utility_f1"]
comparison_df["delta_utility_balanced_accuracy"] = (
    comparison_df["utility_balanced_accuracy"] - baseline_row["utility_balanced_accuracy"]
)
comparison_df["delta_linkability_roc_auc"] = (
    comparison_df["linkability_roc_auc"] - baseline_row["linkability_roc_auc"]
)
comparison_df.sort_values(["delta_linkability_roc_auc", "delta_utility_f1"], ascending=[True, False]).reset_index(drop=True)

,transform_name,method,n_output_features,utility_f1,utility_balanced_accuracy,utility_roc_auc,utility_pr_auc,linkability_f1,linkability_roc_auc,linkability_pr_auc,delta_utility_f1,delta_utility_balanced_accuracy,delta_linkability_roc_auc
0,pca_120,pca,120,0.714832,0.847961,0.927374,0.790344,0.929126,0.982065,0.983659,-0.008735,-0.006271,-0.016316
1,winsor_10_90,winsorization,208,0.721771,0.851521,0.929105,0.793556,0.983229,0.998367,0.998413,-0.001796,-0.002710,-0.000014
2,identity,identity,208,0.723567,0.854232,0.930469,0.794538,0.983976,0.998381,0.998482,0.000000,0.000000,0.000000
3,winsor_05_95,winsorization,208,0.717855,0.848457,0.929945,0.798970,0.983483,0.998587,0.998635,-0.005712,-0.005775,0.000206
4,winsor_01_99,winsorization,208,0.729752,0.857262,0.931998,0.798535,0.983237,0.998797,0.998850,0.006184,0.003030,0.000416


## Save Summary

In [7]:
output_path = OUTPUTS_TABLES_DIR / "privacy_transformations_summary.csv"
comparison_df.to_csv(output_path, index=False)
print(output_path)

C:\Users\Tiago\Documents\GitHub\ecg-privacy\outputs\tables\privacy_transformations_summary.csv
